In [ ]:
%cd ../..

import os
import polars as pl
import numpy as np
import random

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt

from evaluation import *

random.seed(4)
np.random.seed(4)
torch.manual_seed(4)

In [ ]:
embeddings_path = "/data/work/vm/radio-foundation/embeddings/LUNA16"

positives_path = os.path.join(embeddings_path, "nodules")
negatives_path = os.path.join(embeddings_path, "negatives")
positives = [os.path.join(positives_path, p) for p in os.listdir(positives_path)]
negatives = [os.path.join(negatives_path, p) for p in os.listdir(negatives_path)]

labels = {idx:1 for idx in range(len(positives))}
labels.update({idx + len(positives):0 for idx in range(len(negatives))})

id_to_path = {idx : p for idx, p in enumerate(positives)}
id_to_path.update({idx + len(positives) : p for idx, p in enumerate(negatives)})

num_classes = 2

In [ ]:
patient_ids = list(labels.keys())
exist_patient_ids = [x for x in patient_ids if x in id_to_path.keys()]
exist_labels = [labels[pid] for pid in exist_patient_ids]
train_ids, val_ids = train_test_split(exist_patient_ids, test_size=0.2, random_state=5, stratify=exist_labels)

train_dataset = EmbeddingDataset(train_ids, id_to_path, labels, add_noise=True, p=1.0, sigma=0.1)
val_dataset = EmbeddingDataset(val_ids, id_to_path, labels)

In [ ]:
class_weights = [len(exist_labels) / num_classes / exist_labels.count(x) for x in range(num_classes)]
class_weights

In [ ]:
EMBED_DIM = 768
num_epochs = 10
batch_size = 16
learning_rate = 0.001

train_dataloader = DataLoader(
    train_dataset,
    shuffle=True,
    collate_fn=collate_classification,
    batch_size=batch_size,
)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_classification)

device = torch.device("cuda")
pooler = AveragePool()
model = Classifier(pooler, embed_dim=EMBED_DIM).to(device)

loss_fn = torch.nn.BCEWithLogitsLoss()
#loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)

output = train_classifier(
    model,
    optimizer,
    loss_fn,
    train_dataloader,
    val_dataloader,
    num_epochs,
    device
)
model.load_state_dict(output["state_dict"])


In [ ]:
plot_train_curves(output["train_loss"], output["val_loss"], "Cross-Entropy Loss")

In [ ]:
plot_train_curves(output["train_rocauc"], output["val_rocauc"], "ROC AUC")

In [ ]:
all_labels, all_predictions = get_predictions(model, val_dataloader, device)
all_predictions = torch.nn.functional.sigmoid(all_predictions)
all_predictions = all_predictions.flatten() > 0.5

plot_confusion_matrix(all_labels, all_predictions)